# Lab 1: Neural Network Foundations

Capacity can help a network fit its training data, but useful capacity must
also be optimized and must generalize. In this lab you will complete two
small, central pieces of a PyTorch MLP workflow, then make two constrained
experimental choices. Data plumbing, optimization, plotting, and device
handling remain provided so your effort stays on the learning objectives.

This online adaptation is based on MIT 6.7960 Fall 2024 HW1. The original
notebook's two 8,192-unit hidden layers create a 92-million-parameter model.
The default below is much smaller, and the source notebook's broad debugging
burden is replaced by bounded, behavior-tested ownership.

**Learning objectives**

- Build a reusable hidden-layer stack and verify its structural behavior.
- Aggregate cross-entropy and accuracy correctly across arbitrary batch sizes.
- Relate hidden width and depth to parameter count and empirical capacity.
- Distinguish underfitting from overfitting using training and validation curves.
- Test one justified generalization intervention in a controlled comparison.
- Keep the official CIFAR-10 test split out of tuning decisions.


## How this notebook is assessed

Run the visible checks and experiments, then use the plots as evidence for
your own reflection. The MIT Learn submission grades one compact report and
two practitioner decisions. Four report values come from fixed probes or a
workflow contract. One value—your model's parameter count—may vary, but any
design inside the stated range is accepted. **No freely measured CIFAR-10
accuracy or loss is graded** because those values vary across runtimes.

Quick mode is on by default and uses a deterministic 5,000-image training
subset plus a disjoint 1,000-image validation subset. Full mode uses the
entire 50,000-image development set as a 40,000/10,000 split. In both modes,
the official 10,000-image CIFAR-10 test set remains untouched.


In [ ]:
from __future__ import annotations

import math
import random
import time
from dataclasses import asdict, dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 7960
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def reset_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # These settings improve repeatability. Small numerical differences can
    # still occur across devices and library versions, so metrics are ungraded.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


reset_seed()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")


In [ ]:
# Set QUICK_MODE = False for the larger development split and longer runs.
QUICK_MODE = True
DATA_ROOT = "./data"

TRAIN_PER_CLASS = 500 if QUICK_MODE else 4_000
VALIDATION_PER_CLASS = 100 if QUICK_MODE else 1_000
BATCH_SIZE = 256
EXPERIMENT_EPOCHS = 4 if QUICK_MODE else 8
NUM_WORKERS = 0 if QUICK_MODE else 2

print(
    f"quick_mode={QUICK_MODE}, train={10 * TRAIN_PER_CLASS:,}, "
    f"validation={10 * VALIDATION_PER_CLASS:,}, batch={BATCH_SIZE}"
)


## 1. Prepare controlled train and validation splits

CIFAR-10 images are RGB tensors with shape 3×32×32. The source notebook
accidentally mentions 28×28 once, but its code—and this lab—correctly use
32×32 images, so a flattened input has 3,072 values.

The split below is stratified and deterministic. Plain and augmented dataset
views share exactly the same indices. Augmentation is applied **only** to the
training view; validation data receive deterministic normalization only.


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

plain_transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)]
)
augmented_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=3, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)

# This transform-free instance is used only to read class targets.
split_source = datasets.CIFAR10(DATA_ROOT, train=True, download=True)


def stratified_split(
    targets: list[int], train_per_class: int, validation_per_class: int, seed: int
) -> tuple[list[int], list[int]]:
    target_tensor = torch.as_tensor(targets)
    generator = torch.Generator().manual_seed(seed)
    train_indices: list[int] = []
    validation_indices: list[int] = []
    for class_id in range(10):
        class_indices = torch.where(target_tensor == class_id)[0]
        order = torch.randperm(len(class_indices), generator=generator)
        class_indices = class_indices[order]
        needed = train_per_class + validation_per_class
        if needed > len(class_indices):
            raise ValueError(f"Requested {needed} examples from class {class_id}")
        train_indices.extend(class_indices[:train_per_class].tolist())
        validation_indices.extend(
            class_indices[train_per_class:needed].tolist()
        )
    return sorted(train_indices), sorted(validation_indices)


train_indices, validation_indices = stratified_split(
    split_source.targets, TRAIN_PER_CLASS, VALIDATION_PER_CLASS, SEED
)
assert set(train_indices).isdisjoint(validation_indices)

plain_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=plain_transform, download=False
)
augmented_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=augmented_transform, download=False
)

plain_train_set = Subset(plain_source, train_indices)
augmented_train_set = Subset(augmented_source, train_indices)
validation_set = Subset(plain_source, validation_indices)


def seeded_loader(
    dataset,
    *,
    batch_size: int,
    shuffle: bool,
    seed: int = SEED,
) -> DataLoader:
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        generator=generator,
    )


def make_m1_loaders(
    *, use_augmentation: bool, batch_size: int, seed: int = SEED
) -> tuple[DataLoader, DataLoader, DataLoader]:
    fit_set = augmented_train_set if use_augmentation else plain_train_set
    fit_loader = seeded_loader(
        fit_set, batch_size=batch_size, shuffle=True, seed=seed
    )
    train_eval_loader = seeded_loader(
        plain_train_set, batch_size=batch_size, shuffle=False, seed=seed
    )
    validation_loader = seeded_loader(
        validation_set, batch_size=batch_size, shuffle=False, seed=seed
    )
    return fit_loader, train_eval_loader, validation_loader


sample_image, sample_label = plain_train_set[0]
print("training examples:", len(plain_train_set))
print("validation examples:", len(validation_set))
print("sample shape:", tuple(sample_image.shape), "label:", sample_label)
print("official CIFAR-10 test split: reserved and not loaded")


## 2. Student task 1 — build a reusable hidden stack

Complete `build_hidden_stack`. For each requested hidden width, append a
`Linear → ReLU → Dropout` block and update the width that will feed the next
layer. Return both the modules and the final width. The supplied class owns
validation, flattening, the ten-class output layer, and `forward`.

This is deliberately a small construction task: it makes parameter
registration, feature dimensions, and the boundary between architecture and
execution visible without turning the lab into framework debugging.


In [ ]:
def build_hidden_stack(
    input_width: int,
    hidden_dims: tuple[int, ...],
    dropout: float,
) -> tuple[list[nn.Module], int]:
    """Return hidden modules and the width entering the output layer."""
    layers: list[nn.Module] = []
    current_width = input_width

    # STUDENT TASK 1
    raise NotImplementedError(
        "Append one Linear → ReLU → Dropout block per hidden width."
    )

    return layers, current_width


class CompactMLP(nn.Module):
    def __init__(
        self,
        hidden_dims: tuple[int, ...] = (512, 128),
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        if not hidden_dims or any(
            type(width) is not int or width <= 0 for width in hidden_dims
        ):
            raise ValueError(
                "hidden_dims must contain positive integer widths"
            )
        if not 0.0 <= dropout < 1.0:
            raise ValueError("dropout must be in [0, 1)")

        hidden_layers, output_width = build_hidden_stack(
            3 * 32 * 32, hidden_dims, dropout
        )
        self.network = nn.Sequential(
            nn.Flatten(),
            *hidden_layers,
            nn.Linear(output_width, 10),
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.network(images)


In [ ]:
def trainable_parameter_count(model: nn.Module) -> int:
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


def probe_compact_mlp() -> tuple[int, bool]:
    model = CompactMLP(hidden_dims=(7, 5, 3), dropout=0.25).cpu().eval()
    modules = tuple(model.network)

    expected_types = (
        nn.Flatten,
        nn.Linear, nn.ReLU, nn.Dropout,
        nn.Linear, nn.ReLU, nn.Dropout,
        nn.Linear, nn.ReLU, nn.Dropout,
        nn.Linear,
    )
    linear_shapes = [
        (module.in_features, module.out_features)
        for module in modules
        if isinstance(module, nn.Linear)
    ]
    dropout_probabilities = [
        module.p for module in modules if isinstance(module, nn.Dropout)
    ]

    with torch.no_grad():
        output_shape = tuple(
            model(torch.zeros(4, 3, 32, 32)).shape
        )

    contract_passes = (
        tuple(type(module) for module in modules) == expected_types
        and linear_shapes == [(3072, 7), (7, 5), (5, 3), (3, 10)]
        and dropout_probabilities == [0.25, 0.25, 0.25]
        and output_shape == (4, 10)
    )
    return trainable_parameter_count(model), contract_passes


raw_probe_parameter_count, architecture_contract = probe_compact_mlp()
assert architecture_contract, (
    "Check the type and order of every hidden-layer module."
)
assert raw_probe_parameter_count == 21_609
print("Task 1 checks passed.")


## 3. Student task 2 — compute additive batch statistics

Complete `batch_statistics` so it returns summed cross-entropy, the number
of correct predictions, and the number of examples. The evaluator can then
add those quantities across batches before normalizing once over the full
dataset. This matters when the final batch is smaller than the others.

Use raw logits with cross-entropy. Return the loss as a scalar tensor and the
two counts as Python integers; do not mutate the inputs.


In [ ]:
def batch_statistics(
    logits: torch.Tensor,
    labels: torch.Tensor,
) -> tuple[torch.Tensor, int, int]:
    """Return summed cross-entropy, correct predictions, and example count."""
    if logits.ndim != 2 or labels.ndim != 1:
        raise ValueError("Expected [batch, classes] logits and [batch] labels")
    if logits.shape[0] != labels.shape[0]:
        raise ValueError("Logits and labels must have the same batch size")

    # STUDENT TASK 2
    raise NotImplementedError(
        "Compute the summed loss, number correct, and number of examples."
    )


In [ ]:
TOY_PROBABILITIES = torch.tensor(
    [
        [0.7, 0.2, 0.1],
        [0.1, 0.6, 0.3],
        [0.2, 0.5, 0.3],
        [0.1, 0.2, 0.7],
    ],
    dtype=torch.float64,
)
TOY_LOGITS = TOY_PROBABILITIES.log()
TOY_LABELS = torch.tensor([0, 2, 1, 0], dtype=torch.long)

logits_before = TOY_LOGITS.clone()
labels_before = TOY_LABELS.clone()
toy_loss_sum, toy_correct, toy_examples = batch_statistics(
    TOY_LOGITS, TOY_LABELS
)

expected_loss_sum = -math.log(0.7 * 0.3 * 0.5 * 0.1)
assert torch.is_tensor(toy_loss_sum) and toy_loss_sum.ndim == 0
assert abs(toy_loss_sum.item() - expected_loss_sum) < 1e-10
assert toy_correct == 2
assert toy_examples == 4
assert torch.equal(TOY_LOGITS, logits_before)
assert torch.equal(TOY_LABELS, labels_before)
print("Task 2 checks passed.")


## 4. Provided training, evaluation, and plotting rails

Every experiment below rebuilds the model, resets the random seed, and
recreates the shuffled loader. This makes comparisons more controlled. It
does **not** make measured metrics suitable for exact grading across every
runtime.


In [ ]:
@dataclass(frozen=True)
class MLPExperiment:
    hidden_dims: tuple[int, ...] = (512, 128)
    dropout: float = 0.0
    weight_decay: float = 0.0
    use_augmentation: bool = False
    learning_rate: float = 1e-3
    batch_size: int = BATCH_SIZE
    epochs: int = EXPERIMENT_EPOCHS
    seed: int = SEED


@torch.no_grad()
def evaluate_classifier(
    model: nn.Module, loader: DataLoader
) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        loss_sum, correct, examples = batch_statistics(
            model(images), labels
        )
        total_loss += loss_sum.item()
        total_correct += correct
        total_examples += examples
    if total_examples == 0:
        raise ValueError("Cannot evaluate an empty loader")
    return total_loss / total_examples, total_correct / total_examples


def run_mlp_experiment(
    label: str, config: MLPExperiment, *, verbose: bool = True
) -> dict:
    reset_seed(config.seed)
    fit_loader, train_eval_loader, validation_loader = make_m1_loaders(
        use_augmentation=config.use_augmentation,
        batch_size=config.batch_size,
        seed=config.seed,
    )
    model = CompactMLP(config.hidden_dims, config.dropout).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    loss_fn = nn.CrossEntropyLoss()
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "validation_loss": [],
        "validation_accuracy": [],
    }
    started = time.perf_counter()

    for epoch in range(1, config.epochs + 1):
        model.train()
        for images, labels in fit_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()

        train_loss, train_accuracy = evaluate_classifier(model, train_eval_loader)
        validation_loss, validation_accuracy = evaluate_classifier(
            model, validation_loader
        )
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)
        if verbose:
            print(
                f"{label:>18} | epoch {epoch:>2}/{config.epochs} | "
                f"train acc {train_accuracy:6.2%} | "
                f"val acc {validation_accuracy:6.2%}"
            )

    record = {
        "label": label,
        "config": asdict(config),
        "parameter_count": trainable_parameter_count(model),
        "steps_per_epoch": len(fit_loader),
        "seconds": time.perf_counter() - started,
        "history": history,
    }
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record


def plot_history(record: dict) -> None:
    history = record["history"]
    epochs = np.arange(1, len(history["train_accuracy"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
    axes[0].plot(
        epochs, history["validation_loss"], marker="o", label="validation"
    )
    axes[0].set(title=f"{record['label']}: loss", xlabel="epoch", ylabel="loss")
    axes[0].legend()

    axes[1].plot(
        epochs, history["train_accuracy"], marker="o", label="train"
    )
    axes[1].plot(
        epochs,
        history["validation_accuracy"],
        marker="o",
        label="validation",
    )
    axes[1].set(
        title=f"{record['label']}: accuracy",
        xlabel="epoch",
        ylabel="accuracy",
        ylim=(0, 1),
    )
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def plot_sweep(records: list[dict], title: str) -> None:
    """Compare sweep trajectories without relying on color alone."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
    styles = [
        ("-", "o"),
        ("--", "s"),
        ("-.", "^"),
        (":", "D"),
    ]

    for record, (line_style, marker) in zip(records, styles):
        history = record["history"]
        epochs = np.arange(1, len(history["train_accuracy"]) + 1)
        train_accuracy = np.asarray(history["train_accuracy"])
        validation_accuracy = np.asarray(
            history["validation_accuracy"]
        )
        common = {
            "label": record["label"],
            "linestyle": line_style,
            "marker": marker,
        }
        axes[0].plot(epochs, train_accuracy, **common)
        axes[1].plot(epochs, validation_accuracy, **common)
        axes[2].plot(
            epochs,
            train_accuracy - validation_accuracy,
            **common,
        )

    axes[0].set(title="Training accuracy", ylabel="accuracy", ylim=(0, 1))
    axes[1].set(title="Validation accuracy", ylabel="accuracy", ylim=(0, 1))
    axes[2].set(
        title="Train - validation gap",
        ylabel="accuracy difference",
    )
    axes[2].axhline(0, color="black", linewidth=0.8)
    for axis in axes:
        axis.set_xlabel("epoch")
        axis.grid(alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=2,
        frameon=False,
    )
    fig.suptitle(
        f"{title} - single seeded run; measured and ungraded"
    )
    plt.tight_layout(rect=(0, 0.14, 1, 0.93))
    plt.show()


def experiment_summary(records: list[dict]) -> pd.DataFrame:
    rows = []
    for record in records:
        history = record["history"]
        train_accuracy = history["train_accuracy"][-1]
        validation_accuracy = history["validation_accuracy"][-1]
        rows.append(
            {
                "run": record["label"],
                "parameters": record["parameter_count"],
                "steps/epoch": record["steps_per_epoch"],
                "final train acc": train_accuracy,
                "final validation acc": validation_accuracy,
                "train-validation gap": train_accuracy - validation_accuracy,
                "seconds": record["seconds"],
            }
        )
    return pd.DataFrame(rows).set_index("run")


def changed_config_fields(
    reference: MLPExperiment,
    candidate: MLPExperiment,
) -> set[str]:
    reference_values = asdict(reference)
    candidate_values = asdict(candidate)
    return {
        name
        for name in reference_values
        if reference_values[name] != candidate_values[name]
    }


def history_is_complete(record: dict) -> bool:
    expected_epochs = record["config"]["epochs"]
    required_metrics = (
        "train_loss",
        "train_accuracy",
        "validation_loss",
        "validation_accuracy",
    )
    return all(
        len(record["history"].get(metric, ())) == expected_epochs
        for metric in required_metrics
    )


## 5. Establish the shared control

Run the default two-hidden-layer MLP first. Check that loss moves downward,
compare the two accuracy curves, and note whether their gap is widening,
narrowing, or roughly stable. This exact result is reused in both comparisons,
so the notebook trains only four distinct models.


In [ ]:
control_config = MLPExperiment()
low_capacity_config = replace(control_config, hidden_dims=(64,))

control_result = run_mlp_experiment("default control", control_config)
plot_history(control_result)
experiment_summary([control_result])


## 6. Student task 3 — design a capacity comparison

Choose **exactly two positive hidden widths** for a learner-designed model.
Its parameter count must be from 400,000 through 1,000,000, inclusive, and
it must differ from the default control. That range leaves many valid designs;
there is no single intended architecture.

Before editing, predict how your design's training fit and validation behavior
will compare with the supplied one-layer anchor and default control. Then set
`LEARNER_HIDDEN_DIMS`. The checks enforce the experimental boundary while you
retain the design decision.


In [ ]:
# STUDENT TASK 3: replace the empty tuple with exactly two widths.
LEARNER_HIDDEN_DIMS: tuple[int, ...] = ()

if (
    len(LEARNER_HIDDEN_DIMS) != 2
    or any(
        type(width) is not int or width <= 0
        for width in LEARNER_HIDDEN_DIMS
    )
):
    raise ValueError("Choose exactly two positive integer hidden widths.")
if LEARNER_HIDDEN_DIMS == control_config.hidden_dims:
    raise ValueError("Design a model different from the default control.")

learner_capacity_config = replace(
    control_config, hidden_dims=LEARNER_HIDDEN_DIMS
)
learner_parameter_count = trainable_parameter_count(
    CompactMLP(LEARNER_HIDDEN_DIMS)
)
if not 400_000 <= learner_parameter_count <= 1_000_000:
    raise ValueError(
        "Choose widths whose model has 400,000–1,000,000 parameters."
    )
print(f"learner design parameters: {learner_parameter_count:,}")


In [ ]:
low_capacity_result = run_mlp_experiment(
    "low-capacity anchor", low_capacity_config
)
learner_capacity_result = run_mlp_experiment(
    "learner capacity design", learner_capacity_config
)
capacity_results = [
    low_capacity_result,
    control_result,
    learner_capacity_result,
]
plot_sweep(capacity_results, "Capacity comparison")
capacity_table = experiment_summary(capacity_results)
capacity_table


## 7. Student task 4 — predict, choose, and run one intervention

Choose one treatment to compare with the same default control:

1. dropout probability from 0 to 0.30;
2. AdamW weight decay from 0 to (10^{-4}); or
3. train-only reflected crop plus horizontal flip.

Before setting `GENERALIZATION_CHOICE`, predict what will happen to training
accuracy, validation accuracy, and their gap. Select `"dropout"`, `"l2"`, or
`"augmentation"`, then run the comparison. The configuration check permits
exactly one changed field relative to the control.

You do not submit this response. Pause briefly to form a prediction before
viewing the results.


In [ ]:
# STUDENT TASK 4: choose "dropout", "l2", or "augmentation".
GENERALIZATION_CHOICE = ""

TREATMENT_OVERRIDES = {
    "dropout": {"dropout": 0.30},
    "l2": {"weight_decay": 1e-4},
    "augmentation": {"use_augmentation": True},
}
TREATMENT_FIELDS = {
    "dropout": "dropout",
    "l2": "weight_decay",
    "augmentation": "use_augmentation",
}
TREATMENT_LABELS = {
    "dropout": "chosen treatment: dropout 0.30",
    "l2": "chosen treatment: L2 1e-4",
    "augmentation": "chosen treatment: crop + flip",
}

if GENERALIZATION_CHOICE not in TREATMENT_OVERRIDES:
    raise ValueError("Choose dropout, l2, or augmentation.")

chosen_treatment_config = replace(
    control_config,
    **TREATMENT_OVERRIDES[GENERALIZATION_CHOICE],
)
assert changed_config_fields(
    control_config, chosen_treatment_config
) == {TREATMENT_FIELDS[GENERALIZATION_CHOICE]}


In [ ]:
chosen_treatment_result = run_mlp_experiment(
    TREATMENT_LABELS[GENERALIZATION_CHOICE],
    chosen_treatment_config,
)
generalization_results = [control_result, chosen_treatment_result]
plot_sweep(generalization_results, "Generalization interventions")
generalization_table = experiment_summary(generalization_results)
generalization_table


## Pause and reflect — ungraded

Which result most confirmed or changed your prediction? Name the selected run
and identify one visible pattern in the training and validation curves. What
caveat makes that conclusion tentative, and what one-factor experiment would
you try next?

This is a place to pause, not a response field. Your reflection is not graded.


## Further reflection and forum discussion — ungraded

Course staff may return to these questions in the forum. Choose any one to
think about now or discuss later. If you share a response, name the runs you
compared and point to a value or visible curve pattern from your own results.
Do not post code, report values, or answers to the graded decision checks.

1. **Prediction calibration.** What part of your pre-run prediction was
   supported or contradicted? What follow-up would distinguish your explanation
   from ordinary run-to-run variation?
2. **Accuracy versus efficiency.** Find two capacity configurations with
   similar validation behavior but different parameter counts or runtimes.
   Which would you keep under a stated constraint, and what deployment quantity
   did this lab not measure?
3. **Questioning an augmentation assumption.** Crop and horizontal flip assume
   that these image changes preserve the class label. When might that assumption
   fail, and how could you diagnose harm without using the official test split?
4. **Limits of the evidence.** State one conclusion this single seeded split
   supports and one stronger conclusion it does not. What is the smallest
   follow-up that would test stability across seeds, splits, or training budgets?


## Report values

Run the cell below after completing all four student tasks and four training
runs. It prints five compact checks. R1–R3 verify your two implementations on
fixed probes. R4 is the parameter count of **your** valid capacity design, so
different learners may report different values. R5 verifies the disjoint split,
exact configurations, one-factor comparisons, and complete histories.

Enter R1–R5 in the first Lab 1 submission problem in MIT Learn. The accepted
R4 range is 400,000–1,000,000. Measured CIFAR-10 accuracy, loss, and timing are
deliberately absent because they are evidence for interpretation, not exact
grading targets.

**Submission checklist**

- [ ] I entered R1–R5 from the report cell.
- [ ] I answered the widening-gap intervention question.
- [ ] I answered the low-and-close-curves capacity question.
- [ ] I did not upload code or submit a run-specific accuracy as an exact value.

**Provenance:** Adapted from `6_7960_Fall_2024_hw1_CIFAR10.ipynb`,
especially its network, training-loop, learning-curve, and augmentation
sections. Two bounded functions and two constrained choices transfer ownership
to the learner; the network is deliberately smaller, and a true validation
split replaces tuning on the source notebook's test set.


In [ ]:
run_config_pairs = (
    (control_result, control_config),
    (low_capacity_result, low_capacity_config),
    (learner_capacity_result, learner_capacity_config),
    (chosen_treatment_result, chosen_treatment_config),
)

workflow_contract = int(
    set(train_indices).isdisjoint(validation_indices)
    and len({record["label"] for record, _ in run_config_pairs}) == 4
    and all(
        record["config"] == asdict(config)
        for record, config in run_config_pairs
    )
    and all(
        history_is_complete(record)
        for record, _ in run_config_pairs
    )
    and changed_config_fields(control_config, low_capacity_config)
        == {"hidden_dims"}
    and changed_config_fields(control_config, learner_capacity_config)
        == {"hidden_dims"}
    and changed_config_fields(control_config, chosen_treatment_config)
        == {TREATMENT_FIELDS[GENERALIZATION_CHOICE]}
    and learner_capacity_result["parameter_count"]
        == learner_parameter_count
    and 400_000 <= learner_parameter_count <= 1_000_000
)

report_loss_sum, report_correct, report_examples = batch_statistics(
    TOY_LOGITS, TOY_LABELS
)
probe_parameter_count = (
    raw_probe_parameter_count if architecture_contract else -1
)
probe_mean_loss = round(
    report_loss_sum.item() / report_examples, 6
)
probe_accuracy_percent = round(
    100 * report_correct / report_examples
)

report_values = {
    "R1 — fixed probe-model parameter count": probe_parameter_count,
    "R2 — fixed toy-batch mean cross-entropy": probe_mean_loss,
    "R3 — fixed toy-batch accuracy (percent)": probe_accuracy_percent,
    "R4 — learner-designed model parameter count": learner_parameter_count,
    "R5 — experimental workflow contract": workflow_contract,
}
assert probe_parameter_count == 21_609
assert probe_mean_loss == 1.139095
assert probe_accuracy_percent == 50
assert 400_000 <= learner_parameter_count <= 1_000_000
assert workflow_contract == 1

print("LAB 1 REPORT VALUES")
for label, value in report_values.items():
    print(f"{label}: {value}")
